In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# 1. Get the latest code: clone the first time, pull after. Needs Internet ON in the session options.
import os, subprocess
REPO = "/kaggle/working/DOC_Extractor_QUAPT"
if os.path.isdir(REPO + "/.git"):
    print(subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True).stdout)
else:
    r = subprocess.run(["git", "clone", "-b", "feature/prd-v1.3-qwen3-vl-multimodal",
                        "https://github.com/dharicharan/DOC_Extractor_QUAPT.git", REPO],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        raise SystemExit("Clone failed: turn Internet ON in the Kaggle session options and re-run.")
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Install dependencies (needed once per session; safe to re-run)
!pip install -q uv
%cd /kaggle/working/DOC_Extractor_QUAPT
!uv pip install --system -r requirements.txt
!uv pip install --system -e .
!uv pip install --system qwen-vl-utils transformers accelerate bitsandbytes gradio
print("Setup complete")

In [ ]:
# 3. (Optional) Pre-download model weights so the first Extract click is fast.
# Weights are only downloaded here, not loaded onto the GPU: app.py loads them itself.
#  - Qwen3-VL-4B-Instruct: router + default extractor (required)
#  - GLM-OCR + a pure-text model: experimental "GLM-OCR + text LLM" engines (optional)
# If the GLM-OCR engine errors on load, run:  !uv pip install --system -U transformers  then restart and re-run.
from huggingface_hub import snapshot_download

MODELS = [
    ("Qwen/Qwen3-VL-4B-Instruct", True),
    ("zai-org/GLM-OCR", False),
    ("Qwen/Qwen3-4B-Instruct-2507", False),
    ("Qwen/Qwen2.5-7B-Instruct", False),
]
for model_id, required in MODELS:
    try:
        print(f"Downloading {model_id} ...")
        snapshot_download(model_id)
        print("  cached.")
    except Exception as e:
        msg = f"Download failed for {model_id}: {type(e).__name__}. Turn Internet ON in the session options and select a GPU."
        if required:
            raise SystemExit(msg)
        print("  skipped (optional):", msg)

In [ ]:
# 4. Launch the Gradio app (public share link). Needs a GPU in the session options.
!fuser -k 7860/tcp 2>/dev/null || true
%cd /kaggle/working/DOC_Extractor_QUAPT/src/doc_extractor
!python app.py